# 🤖 03 — Customer Segmentation (K-Means Clustering)

> **Objective**: Use unsupervised machine learning to automatically group customers with similar purchasing behavior.

**Pipeline**: StandardScaler -> Elbow Method -> KMeans -> PCA Visualization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
COLORS = {'primary': '#003B73', 'secondary': '#0074B7', 'success': '#27AE60', 'alert': '#BF212F'}
print("Libraries loaded")

## 1. Load RFM Data

In [ ]:
rfm = pd.read_csv('../outputs/rfm_scores.csv')
print(f"Customers: {len(rfm):,}")
features = rfm[['Recency', 'Frequency', 'Monetary']].copy()
features.describe().round(2)

## 2. Feature Scaling

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)
print(f"Scaled features shape: {X_scaled.shape}")
print(f"Mean (should be ~0): {X_scaled.mean(axis=0).round(4)}")
print(f"Std  (should be ~1): {X_scaled.std(axis=0).round(4)}")

## 3. Optimal K — Elbow Method + Silhouette Score

In [ ]:
K_range = range(2, 11)
inertias = []
sil_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(K_range, inertias, 'o-', color=COLORS['primary'], linewidth=2, markersize=8)
axes[0].set_title('Elbow Method', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].axvline(4, color=COLORS['alert'], linestyle='--', alpha=0.7, label='K=4')
axes[0].legend()

axes[1].plot(K_range, sil_scores, 's-', color=COLORS['success'], linewidth=2, markersize=8)
axes[1].set_title('Silhouette Score', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
best_k = list(K_range)[np.argmax(sil_scores)]
axes[1].axvline(best_k, color=COLORS['alert'], linestyle='--', alpha=0.7, label=f'Best K={best_k}')
axes[1].legend()

plt.suptitle('Choosing Optimal K', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../visualizations/elbow_silhouette.png', bbox_inches='tight', dpi=150)
plt.show()
print(f"Best K by Silhouette: {best_k} (score: {max(sil_scores):.4f})")

## 4. Fit Final K-Means Model

In [ ]:
OPTIMAL_K = 4  # Adjust based on elbow/silhouette results
kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(X_scaled)

print(f"Cluster Distribution:")
print(rfm['Cluster'].value_counts().sort_index())
print(f"\nSilhouette Score: {silhouette_score(X_scaled, rfm['Cluster']):.4f}")

## 5. Cluster Profiling

In [ ]:
cluster_profile = rfm.groupby('Cluster').agg(
    Customers=('customer_id', 'count'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Monetary=('Monetary', 'mean'),
    Total_Revenue=('Monetary', 'sum')
).round(2)

# Label clusters based on profile
labels = {}
sorted_by_monetary = cluster_profile.sort_values('Avg_Monetary', ascending=False).index.tolist()
label_names = ['High-Value Customers', 'Mid-Value Active', 'Low-Value Occasional', 'Inactive/Churned']
for i, idx in enumerate(sorted_by_monetary):
    labels[idx] = label_names[min(i, len(label_names)-1)]

cluster_profile['Label'] = cluster_profile.index.map(labels)
rfm['Cluster_Label'] = rfm['Cluster'].map(labels)

print("Cluster Profiles:")
print(cluster_profile.to_string())

## 6. PCA Visualization (2D)

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

rfm['PCA1'] = X_pca[:, 0]
rfm['PCA2'] = X_pca[:, 1]

fig = px.scatter(rfm, x='PCA1', y='PCA2', color='Cluster_Label',
                 hover_data=['customer_name', 'Recency', 'Frequency', 'Monetary'],
                 title='Customer Clusters (PCA 2D Projection)',
                 color_discrete_sequence=px.colors.qualitative.Set2,
                 width=900, height=600)
fig.update_layout(legend_title='Cluster')
fig.show()

In [ ]:
# Static version for export
fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=rfm['Cluster'], cmap='Set2',
                     alpha=0.6, s=30, edgecolors='white', linewidth=0.5)
centers_pca = pca.transform(kmeans.cluster_centers_)
ax.scatter(centers_pca[:, 0], centers_pca[:, 1], c='red', marker='X', s=200,
           edgecolors='black', linewidth=2, zorder=5, label='Centroids')
ax.set_title('Customer Clusters (PCA 2D)', fontsize=15, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.legend()
plt.colorbar(scatter, label='Cluster')
plt.tight_layout()
plt.savefig('../visualizations/customer_clusters.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Cluster radar chart
fig, axes = plt.subplots(1, OPTIMAL_K, figsize=(5*OPTIMAL_K, 5))
metrics = ['Avg_Recency', 'Avg_Frequency', 'Avg_Monetary']
for i, (idx, row) in enumerate(cluster_profile.iterrows()):
    vals = [row[m] for m in metrics]
    axes[i].barh(metrics, vals, color=['#E74C3C', '#3498DB', '#27AE60'])
    axes[i].set_title(f"Cluster {idx}\n{row['Label']}", fontweight='bold')
    axes[i].set_xlim(0, max(cluster_profile[metrics].max()) * 1.1)
plt.suptitle('Cluster Profiles Comparison', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../visualizations/cluster_profiles.png', bbox_inches='tight', dpi=150)
plt.show()

## 7. Export Segments

In [ ]:
export_cols = ['customer_id', 'customer_name', 'segment', 'country', 'Recency', 'Frequency', 'Monetary',
               'R_Score', 'F_Score', 'M_Score', 'RFM_Score', 'Segment', 'Cluster', 'Cluster_Label']
existing = [c for c in export_cols if c in rfm.columns]
rfm[existing].to_csv('../outputs/customer_segments.csv', index=False)
print(f"Exported {len(rfm):,} segmented customers to outputs/customer_segments.csv")

## 8. Key Insights

| # | Insight |
|---|---|
| 1 | K-Means reveals **4 distinct customer groups** with different buying behaviors |
| 2 | **High-Value Customers** have low recency, high frequency, and high monetary |
| 3 | **Inactive/Churned** customers can be targeted with re-engagement campaigns |
| 4 | PCA shows clear cluster separation in 2D space |
| 5 | Combining RFM segments + ML clusters gives the **most actionable segmentation** |

---
*Proceed to `04_Correlation_Analysis.ipynb`*